# 01 — Baseline VLM
Notebook de départ : charger une image, appliquer un prompt et sauvegarder une sortie JSON.

In [27]:
from pathlib import Path
import sys
sys.path.append(str(Path('..').resolve()))
#from src.inference import toy_predict
#from src.guardrails import apply_safety_guardrails
sample = Path('../data/sample_images/CXR_SYN_002_suspected_opacity.png')
#apply_safety_guardrails(toy_predict(sample, mode='baseline'))
import torch
from PIL import Image

In [19]:
def load_image(img_path):

    image = Image.open(img_path).convert("L")

    return image, img_path

image = load_image("../data/test/view1_frontal.jpg")
print(image)
image[0].show("view1_frontal.jpg")

(<PIL.Image.Image image mode=L size=389x320 at 0x202DD62D840>, '../data/test/view1_frontal.jpg')


In [21]:
from transformers import AutoProcessor, AutoModelForMultimodalLM

processor = AutoProcessor.from_pretrained("google/medgemma-1.5-4b-it")
model = AutoModelForMultimodalLM.from_pretrained("google/medgemma-1.5-4b-it", device_map="auto")
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "url": "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/p-blog/candy.JPG"},
            {"type": "text", "text": "What animal is on the candy?"}
        ]
    },
]
inputs = processor.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

print("debugging")

outputs = model.generate(**inputs, max_new_tokens=40)
print(processor.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the disk and cpu.
[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


debugging


KeyboardInterrupt: 

In [38]:
PROMPT = 'You are an educational radiology assistant for engineering students. You are not a clinician and you must not provide a definitive diagnosis. Analyze the provided frontal chest X-ray for a simple educational task: normal vs suspected lung opacity/pneumonia-related abnormality vs uncertain. Return only valid JSON with this schema: { "image_quality": "good | limited | poor", "predicted_class": "normal | suspected_opacity | uncertain", "confidence": 0.0, "visual_evidence": ["short observation 1", "short observation 2"], "justification": "2 to 4 cautious sentences, evidence-based", "limitations": ["possible limitation 1", "possible limitation 2"], "warning": "Educational prototype only. Not for diagnosis. A qualified clinician must verify the image. } Rules: - Do not invent patient history. - Do not mention findings that are not visible. - Use "uncertain" when image quality is poor or evidence is weak. - Avoid definitive clinical diagnosis. - Keep the response concise.'

def predict_chest_xray(image_path : str, prompt = PROMPT):
    image = Image.open(image_path).convert("RGB")

    from transformers import AutoProcessor, AutoModelForMultimodalLM


    model_to_use = "google/medgemma-1.5-4b-it"
    #model_to_use ="microsoft/BioViL-T"
    processor = AutoProcessor.from_pretrained(model_to_use)
    model = AutoModelForMultimodalLM.from_pretrained("google/medgemma-1.5-4b-it", device_map="auto")

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},  # image PIL directement
                {"type": "text", "text": prompt},
            ],
        }
    ]
    inputs = processor.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
    ).to(model.device, dtype=model.dtype)

    input_len = inputs["input_ids"].shape[-1]


    # ✅ model.generate() manquait dans ton code
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=50,
            do_sample=False,
        )

    response = processor.decode(
        outputs[0][input_len:],
        skip_special_tokens=True,
    )
    out = response.strip()
    print(out)
    return out



In [37]:
predict_chest_xray("../data/archive/train/patient00001/study1/view1_frontal.jpg")

OSError: microsoft/BioViL-T is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`

In [ ]:
from random import randint
import pandas as pd

def accuracy_check(nb_of_tests):

    predictions = {}

    for i in range (nb_of_tests):
        idx = randint(0, 64000)
        str_idx = str(idx)

        for i in range(5 - len(str_idx)):
            str_idx = "0" + str_idx

        path_img = "/kaggle/input/datasets/ashery/chexpert/train/patient" + str_idx + "/study1/view1_frontal.jpg"

        label = predict_chest_xray(path_img)

        if "normal" in label :
            predictions[idx] = "normal"
        elif "suspected_opacity" in label :
            predictions[idx] = "suspected_opacity"
        elif "uncertain" in label :
            predictions[idx] = "uncertain"
        else:
            predictions[idx] = "Error"


    #after making all the predictions, we check if they are right.
    df = pd.read_csv("/kaggle/input/datasets/ashery/chexpert/train.csv").fillna(0.0)

    error = 0
    diseases = ["Enlarged Cardiomediastinum","Cardiomegaly","Lung Opacity","Lung Lesion","Edema","Consolidation","Pneumonia","Atelectasis","Pneumothorax","Pleural Effusion","Pleural Other","Fracture"]
    for pred in predictions.keys():
        row = df.iloc[pred]
        if predictions[pred] == "normal":
            if row['No Finding'] != 1.0:

                #debug
                print("label is : normal but shouldn't be")
                print("id of the radio : ", pred)
                #==
                error +=1
        elif predictions[pred] == "suspected_opacity":
            er_temp = True
            for disease in diseases :
                if row[disease] == 1.0 :
                    er_temp = False
                    break
            if er_temp :
                #debug
                print("label is : anormal but shouldn't be")
                #==
                error +=1

        else:  # prédiction : "incertain"
            is_clearly_healthy = row['No Finding'] == 1.0
            is_clearly_sick = any(row[disease] == 1.0 for disease in diseases)

            # Erreur si la vérité terrain est tranchée (clairement sain OU clairement malade)
            if is_clearly_healthy or is_clearly_sick:
                #debug
                print("label is : uncertain but shouldn't be")
                #==
                error += 1

    #after getting the amount of mistakes that have been made, we divide them by the amount of predictions and return it

    return (error/nb_of_tests) * 100

In [ ]:
#test over 60 images
accuracy_check(60)

Output :

label is : normal but shouldn't be
id of the radio :  2397

label is : normal but shouldn't be
id of the radio :  53654

label is : normal but shouldn't be
id of the radio :  29472

label is : normal but shouldn't be
id of the radio :  30876

label is : uncertain but shouldn't be

label is : uncertain but shouldn't be

label is : normal but shouldn't be
id of the radio :  62501

label is : normal but shouldn't be
id of the radio :  14314

label is : uncertain but shouldn't be

label is : normal but shouldn't be
id of the radio :  449

label is : normal but shouldn't be
id of the radio :  8649

label is : normal but shouldn't be
id of the radio :  20254

label is : normal but shouldn't be
id of the radio :  42134

label is : normal but shouldn't be
id of the radio :  56934

label is : normal but shouldn't be
id of the radio :  24755

label is : normal but shouldn't be
id of the radio :  15732

label is : normal but shouldn't be
id of the radio :  54560

label is : normal but shouldn't be
id of the radio :  665

label is : normal but shouldn't be
id of the radio :  50807

label is : normal but shouldn't be
id of the radio :  62337

label is : uncertain but shouldn't be

label is : normal but shouldn't be
id of the radio :  40769

label is : normal but shouldn't be
id of the radio :  43637

label is : normal but shouldn't be
id of the radio :  27357

label is : normal but shouldn't be
id of the radio :  48588

label is : anormal but shouldn't be

label is : normal but shouldn't be
id of the radio :  12813

label is : normal but shouldn't be
id of the radio :  53405

label is : normal but shouldn't be
id of the radio :  24108

label is : normal but shouldn't be
id of the radio :  31449

label is : normal but shouldn't be
id of the radio :  47620

label is : normal but shouldn't be
id of the radio :  30042

label is : normal but shouldn't be
id of the radio :  36273

label is : normal but shouldn't be
id of the radio :  53056

label is : normal but shouldn't be
id of the radio :  41377

label is : normal but shouldn't be
id of the radio :  8280

label is : normal but shouldn't be
id of the radio :  15226

label is : anormal but shouldn't be

label is : anormal but shouldn't be

label is : normal but shouldn't be
id of the radio :  56235

label is : anormal but shouldn't be

label is : normal but shouldn't be
id of the radio :  57116

label is : normal but shouldn't be
id of the radio :  8949

label is : normal but shouldn't be
id of the radio :  21830

label is : uncertain but shouldn't be

label is : uncertain but shouldn't be


final accuracy :  76.66666666666667 %